# MDR Phase 2 — Unlimited-OCR Batch (Colab GPU)

Recovers the 40 degraded (scanned-look) PDFs back into text.

**Before running:**
1. In your Google Drive, create a folder `MDR_Phase2` and upload `degraded.zip`
   (or the raw `corpus/degraded/` folder) into it. A zip is auto-extracted in step 2
   below — no manual unzip needed.
2. `Runtime` → `Change runtime type` → GPU (T4 / L4 / A100, whichever is available).
3. Run all cells top to bottom.

**After running:** download `ocr_output/` and `ocr_run_log.json` from
`MyDrive/MDR_Phase2/` back into the repo's `corpus/` folder, then continue with Phase 2
(entity integrity + metrics).

This notebook reuses gotchas already discovered while running Unlimited-OCR locally:
`transformers==4.57.1` is pinned (5.x removed `is_torch_fx_available`, which the model's
remote code imports), and the model is loaded in `bfloat16` (its remote code hardcodes
image tensors to bf16). Colab gives a real CUDA GPU, so — unlike the earlier Apple-MPS
run — no `.cuda()` monkeypatching is needed here.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths & sanity-check the upload

Uploaded a `degraded.zip` instead of the raw folder? This cell auto-extracts it —
no manual unzip step needed.

In [ ]:
import zipfile
from pathlib import Path

# Change this if you uploaded degraded/ (or degraded.zip) somewhere else in your Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive/MDR_Phase2')
DEGRADED_DIR = DRIVE_ROOT / 'degraded'
DEGRADED_ZIP = DRIVE_ROOT / 'degraded.zip'
OCR_OUTPUT_DIR = DRIVE_ROOT / 'ocr_output'
RUN_LOG_PATH = DRIVE_ROOT / 'ocr_run_log.json'

OCR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(DEGRADED_DIR.glob('*_scan.pdf'))
if len(pdfs) < 40 and DEGRADED_ZIP.exists():
    print(f'Extracting {DEGRADED_ZIP} ...')
    with zipfile.ZipFile(DEGRADED_ZIP) as zf:
        zf.extractall(DRIVE_ROOT)
    pdfs = sorted(DEGRADED_DIR.glob('*_scan.pdf'))

print(f'Found {len(pdfs)} degraded PDFs in {DEGRADED_DIR}')
assert len(pdfs) == 40, (
    f'Expected 40 degraded PDFs, found {len(pdfs)} — check DRIVE_ROOT and that the '
    'upload/extraction finished.'
)

## 3. Confirm GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'No GPU detected — set Runtime > Change runtime type > GPU and re-run.'
)
gpu_name = torch.cuda.get_device_name(0)
print(f'GPU: {gpu_name}')
print(f'torch: {torch.__version__}')

## 4. Install pinned dependencies

`torch`/`torchvision` ship preinstalled and CUDA-matched on Colab GPU runtimes —
we never touch `torch` itself, and only install `torchvision` if it's somehow
missing, to avoid pip silently swapping in a mismatched CPU build.

In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)


# transformers must be pinned: 5.x removed is_torch_fx_available, which the
# Unlimited-OCR remote code (modeling_deepseekv2.py) imports at load time.
pip_install('transformers==4.57.1', 'accelerate')

# Hidden deps the remote code imports but doesn't declare — discovered by trial and
# error running this model locally; installed only if not already present.
_hidden_deps = [
    ('addict', 'addict'),
    ('easydict', 'easydict'),
    ('einops', 'einops'),
    ('tiktoken', 'tiktoken'),
    ('matplotlib', 'matplotlib'),
    ('requests', 'requests'),
    ('pymupdf', 'fitz'),
]
for pkg, import_name in _hidden_deps:
    if importlib.util.find_spec(import_name) is None:
        pip_install(pkg)

if importlib.util.find_spec('torchvision') is None:
    pip_install('torchvision')

print('Dependencies ready.')

## 5. Load Unlimited-OCR

In [ ]:
from transformers import AutoModel, AutoTokenizer

OCR_MODEL = 'baidu/Unlimited-OCR'

tokenizer = AutoTokenizer.from_pretrained(OCR_MODEL, trust_remote_code=True)
model = AutoModel.from_pretrained(
    OCR_MODEL,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)
model = model.eval().cuda()
print('Unlimited-OCR loaded on', next(model.parameters()).device)

## 6. Page rendering helper

In [ ]:
import fitz  # PyMuPDF — pure-Python wheel, no system poppler needed on Colab


def render_pdf_to_images(pdf_path: Path, out_dir: Path, dpi: int = 200) -> list[str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    paths = []
    with fitz.open(pdf_path) as doc:
        for i, page in enumerate(doc):
            pix = page.get_pixmap(matrix=matrix)
            img_path = out_dir / f'page_{i:03d}.png'
            pix.save(img_path)
            paths.append(str(img_path))
    return paths

## 7. Run the batch

Resume-safe: re-running this cell skips any `doc_id` that already has output in
`ocr_output/`, and a per-doc failure is logged without stopping the batch — both
matter here since a 40-document GPU batch can outlast a Colab session.

In [ ]:
import json
import tempfile
import time

run_log = {'model': OCR_MODEL, 'gpu': gpu_name, 'torch_version': torch.__version__, 'docs': {}}
if RUN_LOG_PATH.exists():
    run_log = json.loads(RUN_LOG_PATH.read_text())
    run_log.setdefault('docs', {})

for pdf_path in pdfs:
    doc_id = pdf_path.stem.replace('_scan', '')
    out_md = OCR_OUTPUT_DIR / f'{doc_id}.md'
    if out_md.exists():
        print(f'[skip] {doc_id} already has output')
        continue

    t0 = time.time()
    try:
        with tempfile.TemporaryDirectory() as tmp:
            tmp_path = Path(tmp)
            image_paths = render_pdf_to_images(pdf_path, tmp_path)
            result = model.infer_multi(
                tokenizer,
                prompt='<image>Multi page parsing.',
                image_files=image_paths,
                output_path=str(tmp_path / 'ocr_out'),
                image_size=1024,
                max_length=32768,
                no_repeat_ngram_size=35,
                ngram_window=1024,
                save_results=True,
            )
            if isinstance(result, str) and result.strip():
                text = result.strip()
            else:
                out_dir = tmp_path / 'ocr_out'
                parts = []
                for pattern in ('*.mmd', '*.md', '*.txt'):
                    for f in sorted(out_dir.glob(pattern)):
                        parts.append(f.read_text(encoding='utf-8', errors='ignore'))
                text = '\n\n'.join(parts).strip()

        out_md.write_text(text)
        elapsed = time.time() - t0
        run_log['docs'][doc_id] = {'status': 'ok', 'seconds': round(elapsed, 2), 'chars': len(text)}
        print(f'[ok] {doc_id}: {len(text)} chars in {elapsed:.1f}s')

    except Exception as exc:
        elapsed = time.time() - t0
        run_log['docs'][doc_id] = {'status': 'error', 'seconds': round(elapsed, 2), 'error': str(exc)}
        print(f'[FAIL] {doc_id}: {exc}')

    RUN_LOG_PATH.write_text(json.dumps(run_log, indent=2))

ok = sum(1 for d in run_log['docs'].values() if d['status'] == 'ok')
print(f'Done: {ok}/{len(pdfs)} succeeded')

## 8. Done — sync back to the repo

Download these from `MyDrive/MDR_Phase2/` and place into the repo's `corpus/` folder:
- `ocr_output/` → `corpus/ocr_output/`
- `ocr_run_log.json` → `corpus/ocr_run_log.json`

Then continue with Phase 2 (entity integrity + metrics).